In [ ]:
import os, sys
HOME = os.environ["HOME"]
CARDIAC_COMA_REPO = f"{HOME}/01_repos/CardiacCOMA/"
CARDIAC_MOTION_REPO = f"{HOME}/01_repos/CardiacMotion/"

In [ ]:
import mlflow

import torch
import torch.nn.functional as F

os.chdir(CARDIAC_COMA_REPO)
from config.load_config import load_yaml_config, to_dict

import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import Image
from mlflow.tracking import MlflowClient

import pickle as pkl
import pytorch_lightning as pl

from argparse import Namespace
import matplotlib.pyplot as plt

#import surgeon_pytorch
#from surgeon_pytorch import Inspect, get_layers

import numpy as np
import pandas as pd
from IPython import embed
sys.path.insert(0, '..')

import model.Model3D
from utils.helpers import get_coma_args, get_lightning_module, get_datamodule
from pprint import pprint

from copy import deepcopy
from typing import List

In [ ]:
from mlflow_helpers import \
    list_artifacts,\
    get_significant_loci,\
    get_metrics_cols, \
    get_params_cols, \
    get_runs_df, \
    get_good_runs,\
    summarize_loci_across_runs,\
    get_model_pretrained_weights

In [ ]:
TRACKING_URI = f"file://{CARDIAC_COMA_REPO}/mlruns"
mlflow.set_tracking_uri(TRACKING_URI)

In [ ]:
client = MlflowClient()

### Retrieve cardiac indices

In [ ]:
# timeframe = "1".zfill(3)
# datafolder = "data/cardio/cardiac_indices"
# df = pd.concat([pd.read_csv(f"{datafolder}/G{i}/LVRV_time{timeframe}.csv", index_col="case_id") for i in range(1,5)])

In [ ]:
import pyvista as pv
from ipywidgets import interact, interactive, fixed, interact_manual

In [ ]:
cardiac_indices_df = pd.read_csv("/home/rodrigo/01_repos/CardiacCOMA/data/cardio/cardiac_indices/cardiac_indices.csv", index_col="ID")
cardiac_indices_df

In [ ]:
def experiment_selection_widget():
    options = [exp.name for exp in mlflow.list_experiments()]

    experiment_w = widgets.Select(
      options=options,
      value="Cardiac - ED"
    )
    
    return experiment_w

exp_w = experiment_selection_widget()

@interact
def get_runs(exp_name=exp_w):  
  try:
    exp_id = mlflow.get_experiment_by_name(exp_name).experiment_id
    runs_df = get_runs_df(exp_name=exp_name, only_finished=True)
    metrics, params = get_metrics_cols(runs_df), get_params_cols(runs_df)  
    # display(runs_df.loc[:, [*metrics, *params]].drop("params.platform", axis=1).head(10))
    return runs_df
  except:
    pass

In [ ]:
runs_df = get_runs_df(exp_name=exp_w.value, only_finished=True)

In [ ]:
exp_id = "2"
run_id = "6a4d73fb59f24d97b37764afdedd4185"

In [ ]:
z_df = pd.read_csv(f"{CARDIAC_MOTION_REPO}/mlruns/{exp_id}/{run_id}/artifacts/output/latent_vector.csv")
z_df = z_df.set_index("ID")

In [ ]:
z_corr_df = z_df.corr().abs()

In [ ]:
z_corr_df

## Correlation between latent variables

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
corr_lst = np.triu(z_corr_df.to_numpy()).flatten()

In [ ]:
plt.hist(corr_lst[~(corr_lst == 1.) & ~(corr_lst == 0.)], bins=20);

___

## Correlation between latent variables and cardiac indices

In [ ]:
z_df = z_df.set_index(z_df.index.astype(int))

In [ ]:
common_subjects = list(set(cardiac_indices_df.drop_duplicates().index).intersection(set(z_df.drop_duplicates().index)))

In [ ]:
len(common_subjects)

In [ ]:
ccii = cardiac_indices_df.loc[sorted(common_subjects)].drop_duplicates()
zz = z_df.loc[sorted(common_subjects)].drop_duplicates()

In [ ]:
ccii.corrwith(zz, axis=1)

In [ ]:
ccii.isna().count()

In [ ]:
ccii.to_csv("lvedv_lvm_rvedv_lvsph.csv", index=True, index_label="ID")

In [ ]:
corr_matrix = pd.concat([zz, ccii], axis=1).corr() #.index.str.startswith("LV")

In [ ]:
z_per_loci = { 
    "64bde_z003": "PLN",
    "25cd6_z009": "GOSR2",
    "e93c4_z008": "TTN",
    "06413_z004": "TBX5",
    "0285f_z007": "NKX2.5",
    "383a4_z003": "LMO7",
    "8b630_z003": "RBM20",
    "9a924_z002": "VPS37A",
    "e6490_z012": "WAC",
    "cc438_z002": "LGALS8",
    "64bde_z000": "CCDC91",
    "d02d6_z011": "EN1",
    "b0ca2_z004": "BAG3",
    "cc438_z003": "OR9Q1",
    "28a33_z006": "STRN"
}

In [ ]:
corr_df = corr_matrix.loc[:, ["LVEDV", "LVESV", "LVEDSph", "LVM", "LVMVR", "LVEF", "LVSV"]]

In [ ]:
corr_df

In [ ]:
corr_df.index.rename(z_per_loci.values())

In [ ]:
corr_df.index = z_per_loci.values()

In [ ]:
open("corr_with_cardiac_indices.tex", "wt").write(corr_df.to_latex(float_format="%.3f"))

In [ ]:
z_corr = zz.corr()

In [ ]:
corr_z_vs_indices = corr_matrix[ccii.columns]

In [ ]:
corr_z_vs_indices.to_csv("data/cardio/corr_z_vs_indices.csv", index=True, index_label="phenotype")

In [ ]:
len([ind for ind in cardiac_indices_df.index if ind in z_all_df.index])

In [ ]:
cardiac_indices_df.index

In [ ]:
pd.read_csv("data/cardio/corr_z_vs_indices.csv")